# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")
print(f"Version: {metadata_json['version']}")
print(f"Published: {metadata_json['datePublished']}")
print("Available keywords:", metadata_json['keywords'])

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema organizes tabular data through record sets and fields. Here we enumerate all top-level record sets and their fields by `@id`.

In [ ]:
# Retrieve record sets from metadata
record_sets = dataset.metadata.record_sets
print("Found record sets:")
for rs in record_sets:
    print(f"- Record set: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    print("  Fields:")
    for field in rs.get('fields', []):
        print(f"    - Field: {field['@id']} | Name: {field.get('name', 'N/A')} | DataType: {field.get('dataType', 'N/A')}")
    print()
# Preview a sample record from the first record set
if len(record_sets) > 0:
    sample_rs_id = record_sets[0]['@id']
    for record in dataset.records(record_set=sample_rs_id):
        print(f"Sample record from {sample_rs_id}:")
        print(record)
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s obtained in the previous section.

In [ ]:
# Extract data from each record set
# Build the list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
        else:
            df = pd.DataFrame()
        dataframes[record_set_id] = df
        print(f"Record set {record_set_id} loaded. Columns:", df.columns.tolist())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Show preview of the first non-empty record set
for record_set_id, df in dataframes.items():
    if not df.empty:
        print(f"\nData preview for record set: {record_set_id}")
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations may include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify which record set to use and which fields are numeric
chosen_rs_id = None
numeric_field_id = None
group_field_id = None

# Search for numeric fields
for rs in dataset.metadata.record_sets:
    fields = rs.get('fields', [])
    for field in fields:
        if field.get('dataType', '').lower() in ['integer', 'float', 'number']:
            chosen_rs_id = rs['@id']
            numeric_field_id = field['@id']
            break
    if chosen_rs_id:
        break
# Choose a group field (categorical)
if chosen_rs_id:
    for field in dataset.metadata.record_sets[0].get('fields', []):
        if field.get('dataType', '').lower() == 'text':
            group_field_id = field['@id']
            break

print(f"Using record set: {chosen_rs_id}")
print(f"Numeric field: {numeric_field_id}")
print(f"Group field: {group_field_id}")

# Filter, normalize, and group
if chosen_rs_id and not dataframes[chosen_rs_id].empty:
    df = dataframes[chosen_rs_id]
    # Convert numeric field to numeric dtype (if possible)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a categorical field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped average {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot the distribution of the numeric field and compare group means.

In [ ]:
# Visualization
if chosen_rs_id and not dataframes[chosen_rs_id].empty:
    df = dataframes[chosen_rs_id]
    if numeric_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(7,4))
        df[numeric_field_id].dropna().hist(bins=10)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()
    # Grouped bar plot
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        group_means.plot(kind='bar', figsize=(8,5))
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook provided a walkthrough of loading, inspecting, filtering, and visualizing tabular clinical and molecular data of second primary colorectal cancer survivors, using the `mlcroissant` standard and referencing all fields by their `@id`.

- The dataset is well-structured with clear record sets and field types, enabling robust extraction and exploration.
- Numeric fields (such as those capturing age, intervals, or biomarker counts) can be filtered and normalized for downstream analysis.
- Categorical grouping (e.g., anatomical location or diagnosis type) reveals potential differences in clinical or molecular characteristics.
- Visualizations help to quickly understand distributions and relationships among key features.

Always refer to dataset documentation and ethical statements when processing clinical data.